# H02 — Linear systems: the oscillator a detector is made of

**Part IV · the laboratory that follows lecture 8: tasks 1 to 9 in class, tasks 10 to 18 on your own**

> **References**
>
> (Saulson 2017), (Oppenheim et al. 1997)

## What you do here

A transfer function tells us what a system does to any input, without solving its equation again. This laboratory builds one, shows why it is useful and puts it to work. Lecture 8 gave the route. We follow it on the damped oscillator, in small tasks. Each task has one idea and needs only the tasks before it.

| Part | The question | Tasks |
|---|---|---|
| A · The oscillator the hard way | What does the oscillator do when we integrate its equation? | 1 to 3 |
| B · A linear system is its impulse response | Can one kick give the output for every drive? | 4 to 6 |
| C · The transfer function | Can we find the output without integrating? | 7 to 9 |
| D · Stability | What do the poles say about the motion? | 10 |
| E · The Bode plot and the resonance | How is a transfer function drawn and measured? | 11 and 12 |
| F · Laplace or Fourier | Which transform answers which question? | 13 and 14 |
| G · Applications | Control, filters and the suspension of a mirror | 15 to 18 |

**In class we stop after Part C, at task 9.** Parts D to G are for your own time. You can also stop at any task. The next one starts from what the last one showed.

The oscillator is a mirror on its suspension. Its transfer function is

$$
G(s) = \frac{\omega_{0}^{2}}{s^{2} + \dfrac{\omega_{0}}{Q}\,s + \omega_{0}^{2}} .
$$

> **The symbols**
>
> - **$x(t)$** — the input: the motion of the support
> - **$y(t)$** — the output: the motion of the mass
> - **$s$** — the complex frequency of the Laplace transform, in 1/s
> - **$\omega_{0} = 2\pi f_{0}$** — the undamped angular resonance, in radians per second
> - **$Q$** — the quality factor: how many radians of oscillation fit into one e-folding of the energy
> - **$G(s)$** — the transfer function: the output over the input, in the $s$ plane

Nothing here needs data. Every curve is computed from the equation of the oscillator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import solve_ivp
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

# the oscillator of tasks 1 to 17: change these
F0 = 1.0          # resonance, Hz
Q = 30.0          # quality factor

# the line the notch of tasks 8 and 17 removes
F_LINE = 50.0     # Hz, the mains
Q_NOTCH = 10.0    # quality factor of the notch

# the suspension of task 18: change these too
F_PENDULUM = 0.6  # resonance of one pendulum stage, Hz
PHI = 1e-3        # loss angle of the suspension fibres
N_STAGES = 4      # how many stages hang one below the other

W0 = 2 * np.pi * F0
TAU0 = 2 * Q / W0                     # amplitude e-folding time, s

# the random drive of tasks 6 and 9: 20 s of Gaussian numbers, rms 1, at 200 Hz
FS = 200.0
T_NOISE = np.arange(0, 20, 1 / FS)
X_NOISE = np.random.default_rng(20150914).normal(0.0, 1.0, T_NOISE.size)

print(f"f0 = {F0} Hz, Q = {Q}, the ringing dies by a factor e in {TAU0:.3f} s")

> **Try these before you read the answers**
>
> `F0` and `Q` at the top control tasks 1 to 17. Change `Q` once the notebook runs
> and see which results move and which do not. Write each task in the empty cell
> first. Open the solution only to compare. Some tasks use a function or an array
> from an earlier one. The line **Needs** says which.

## Part A · The oscillator the hard way

First we do what Lecture 8 lets us avoid. We integrate the equation of the oscillator in time and measure what it does.

### Task 1 · The free decay

Lecture 8 wrote the oscillator as one second-order equation. A solver wants first-order equations, so we split it in two.

**Goal.** Integrate the oscillator with no drive and draw the ringing inside its envelope.

The equation of lecture 8, divided by the mass $m$ and with $b/m = \omega_{0}/Q$, is

$$
\ddot{y} + \frac{\omega_{0}}{Q}\,\dot{y} + \omega_{0}^{2}\,y = \omega_{0}^{2}\,x(t) .
$$

**Steps.**

1. Write a function `rhs(t, state, drive)`. The state is `(y, v)` with $v = \dot{y}$. It returns the list `[dy/dt, dv/dt]`. The second entry comes from the equation above. The drive is a function: use `drive(t)` for $x(t)$.
2. Write `no_drive(t)` that returns `0.0`. Integrate with `solve_ivp` from $y = 1$ and $v = 0$ for 20 s, with `args=(no_drive,)`. Sample at 400 Hz with `t_eval`. Use `rtol=1e-10` and `atol=1e-12`.
3. Plot $y(t)$ and the two curves $\pm\mathrm{e}^{-t/\tau_{0}}$, with $\tau_{0}$ = `TAU0`.
4. Print $\tau_{0}$. Print the largest value of $|y|\,\mathrm{e}^{t/\tau_{0}}$.

> **The symbols**
>
> - **$v(t)$** — the velocity of the mass, $\dot{y}$
> - **`state`** — the pair $(y, v)$: all `solve_ivp` needs to know at one instant
> - **$\tau_{0} = 2Q/\omega_{0}$** — the decay time of the amplitude

**What to look for.** The mass rings and the peaks sit on the envelope. The number of step 4 is close to 1, so the envelope is the decay $\mathrm{e}^{-t/\tau_{0}}$. In the $s$ plane this rate is the real part of the poles: $\mathrm{Re}\,p = -1/\tau_{0}$.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 2 · The ringing frequency

The envelope says how fast the ringing dies. The zero crossings say how fast it rings.

**Goal.** Measure the frequency of the ringing and compare it with the damped frequency of lecture 8.

**Needs.** `t` and `y` from task 1.

**Steps.**

1. Find the samples where `y` changes sign. Between the two samples around each change, draw a straight line and take the time where it reaches zero. These are the zero crossings.
2. Two crossings in a row are half a period apart. Take the mean of the spacings. Print the frequency `f_measured` with six decimals.
3. Print $f_{d} = f_{0}\sqrt{1 - 1/4Q^{2}}$ and $f_{0}$ with six decimals.
4. Solve the formula of $f_{d}$ for $Q$. Print the $Q$ for which $f_{d}$ is 1% below $f_{0}$.

> **The symbols**
>
> - **$f_{d} = \omega_{d}/2\pi$** — the damped frequency: the frequency of the imaginary part of the poles

**What to look for.** The measured frequency agrees with $f_{d}$ to six digits. It knew nothing of the formula. At $Q = 30$ the shift from $f_{0}$ is one part in $10^{4}$. You need a $Q$ of about 3.5 to move the frequency by 1%.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 3 · The energy and $Q$

The amplitude falls by $\mathrm{e}$ in $\tau_{0}$. The energy is the square of the amplitude, so it falls twice as fast. That gives a way to measure $Q$.

**Goal.** Recover $Q$ from the decay of the energy.

**Needs.** `t`, `y` and `v` from task 1.

**Steps.**

1. Compute the energy per unit mass, $E = \tfrac{1}{2}v^{2} + \tfrac{1}{2}\omega_{0}^{2}y^{2}$.
2. Fit a straight line to $\ln E$ against $t$ between 1 s and 15 s with `np.polyfit`. Plot $E$ on a logarithmic axis with the fitted line on top.
3. The energy falls as $\mathrm{e}^{-2t/\tau_{0}} = \mathrm{e}^{-\omega_{0}t/Q}$. So the slope of $\ln E$ is $-\omega_{0}/Q$. Print $Q$ from the slope. Print the e-folding time of the energy next to $Q/\omega_{0}$.
4. In one cycle the time grows by $2\pi/\omega_{0}$. Print the fraction of the energy lost in one cycle, $1 - \mathrm{e}^{-2\pi/Q}$, next to $2\pi/Q$.

> **The symbols**
>
> - **$E$** — the energy of the mass per unit mass: kinetic plus spring

**What to look for.** The fit returns $Q$ close to the 30 that went in. The straight line on the logarithmic axis says that the loss per radian is the same at every amplitude. The number $2\pi/Q$ is the fraction of the energy lost per cycle when $Q$ is large. At $Q = 30$ it is a little too big. A suspension with $Q = 10^{6}$ loses about one part in $10^{5}$ per swing. That number sets its thermal noise.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Part B · A linear system is its impulse response

Part A integrated the equation for one start. Part B asks whether one special start, a kick, is enough to know the output for every drive.

### Task 4 · The kick

A drive that lasts one instant is the building block of every other drive. We kick the oscillator and watch.

**Goal.** Compute the impulse response $g(t)$ and compare it with the formula of lecture 8.

**Needs.** `rhs` and `no_drive` from task 1.

**Steps.**

1. A unit impulse in $x(t)$ adds $\omega_{0}^{2}$ to $v$ at $t = 0$, because $x$ enters the equation with the factor $\omega_{0}^{2}$. So integrate with no drive from $y = 0$ and $v = \omega_{0}^{2}$. Use the grid `T_NOISE` and the same tolerances as in task 1. Call the result `g_kick`.
2. Plot `g_kick` against time.
3. Compute $g(t) = \dfrac{\omega_{0}^{2}}{\omega_{d}}\,\mathrm{e}^{-t/\tau_{0}}\sin\omega_{d}t$ on the same grid. Print the largest difference from `g_kick` and the largest value of $g$.

> **The symbols**
>
> - **$g(t)$** — the impulse response: the output for a unit impulse in the input
> - **$\omega_{d} = \omega_{0}\sqrt{1 - 1/4Q^{2}}$** — the damped angular frequency

**What to look for.** The two curves are the same. The impulse response is the free ringing that a kick starts. Lecture 8 says that $g$ describes the whole system. The next two tasks test that.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 5 · Superposition

The impulse response describes the system only if the system is linear. Then the output for a sum of drives is the sum of the outputs.

**Goal.** Check that the oscillator answers two sinusoids at once as the sum of its two answers.

**Needs.** `rhs` from task 1.

**Steps.**

1. Define three drives. Drive `a` is $\cos(2\pi\,0.7\,t)$. Drive `b` is $0.5\cos(2\pi\,2.3\,t)$. Drive `ab` is their sum.
2. Write `run(drive)` that integrates from rest for 40 s on a grid at 200 Hz and returns $y$. Use `rtol=1e-10` and `atol=1e-12`. Call it for the three drives.
3. Print the largest value of $|y_{ab} - y_{a} - y_{b}|$ and the largest value of $|y_{ab}|$.
4. Plot $y_{ab}$ and the sum $y_{a} + y_{b}$ on one axis. Draw the sum as a dashed line.

**What to look for.** The two curves lie on top of each other. The difference is set by the tolerance of the solver. The equation has no term that could make anything else. This is why we may cut any drive into pieces.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 6 · A random drive, two ways

Because the system is linear we can cut any drive into kicks. The sum of the answers to all the kicks is a convolution.

**Goal.** Compute the output for a noise drive by integration and as the convolution of $g$ with the drive.

**Needs.** `rhs` from task 1 and `g_kick` from task 4.

**Steps.**

1. Integrate `rhs` from rest for 20 s with the drive `X_NOISE`. The drive function is `np.interp(t, T_NOISE, X_NOISE)`. Use `t_eval=T_NOISE`, `rtol=1e-8`, `atol=1e-10` and `max_step=1/FS`. Call the output `y_ivp`.
2. Compute `y_conv` as `np.convolve(g_kick, X_NOISE)` times $\Delta t = 1/f_{s}$, keeping the first samples as long as `T_NOISE`.
3. Plot both outputs on one axis. Draw the convolution as a dashed line.
4. Print the rms of `y_ivp` and the largest difference divided by that rms.

> **The symbols**
>
> - **$\Delta t = 1/f_{s}$** — the time step of the samples. The integral of the convolution becomes a sum times $\Delta t$
> - **rms** — the root mean square of a signal about its mean

**What to look for.** The convolution gives the output of the integration. It never solved the equation. The small difference is the rectangle rule of `np.convolve`. It falls when the sampling rate rises. One function $g(t)$ gives the output for any drive.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Part C · The transfer function

Part B needed $g$. It came from an integration. Lecture 8 gives a shortcut: $G(s)$ is a ratio of polynomials that we can write down. Part C reads the motion from it and then finds the output without any integration.

### Task 7 · $G(s)$ and its poles

The kick gave us $g(t)$ as a curve. The same information is in the polynomials of $G(s)$. We read it there.

**Goal.** Write $G(s)$ as a ratio of two polynomials, find its poles and draw them in the $s$ plane.

**Steps.**

1. Write a function `oscillator(q)` that returns the `TransferFunction` of $G(s)$ above. The numerator is the list `[W0**2]`. The denominator is the list of the coefficients of $s^{2}$, $s$ and $1$. Later tasks use this function.
2. For $Q = 0.5$, $2$, $10$ and $30$ read the poles from `.poles`. Print the first pole of each, the ratio $|p|/\omega_{0}$ and the number of zeros.
3. Compare the poles with the formula of lecture 8, $p_{\pm} = -\dfrac{\omega_{0}}{2Q} \pm \mathrm{i}\,\omega_{0}\sqrt{1 - 1/4Q^{2}}$. Print the largest difference.
4. Draw the four pairs of poles as crosses in the $s$ plane: $\mathrm{Re}\,s$ on the horizontal axis and $\mathrm{Im}\,s$ on the vertical one. Add the circle $|s| = \omega_{0}$.

> **The symbols**
>
> - **$p_{\pm}$** — the two poles of the oscillator, a complex conjugate pair for $Q > 1/2$

**What to look for.** The four pairs lie on the circle. As $Q$ grows the poles slide along it towards the imaginary axis. At $Q = 1/2$ the two meet on the real axis. The oscillator has no zeros. The real part of a pole is the decay rate that you saw in task 1.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 8 · Zeros: a notch

The oscillator has poles and no zeros. Zeros are the other half of a transfer function. A zero on the imaginary axis removes one frequency.

**Goal.** Build a transfer function with zeros and see what they do.

**Steps.**

1. Write `notch(f_zero, q)` that returns the `TransferFunction`
$$
G(s) = \frac{s^{2} + \omega_{z}^{2}}{s^{2} + (\omega_{z}/Q_{n})\,s + \omega_{z}^{2}} .
$$
The numerator is not a constant now: it is the list `[1, 0, wz**2]`. Build the notch for `F_LINE` and `Q_NOTCH`. Task 17 uses it.
2. Print its zeros and its poles.
3. Draw the zeros as circles and the poles as crosses in the $s$ plane.
4. Print $|G(\mathrm{i}\omega)|$ at $\omega_{z}$ and one decade below and above it, with `signal.freqresp`.

> **The symbols**
>
> - **$\omega_{z} = 2\pi f_{z}$** — the angular frequency of the two zeros of the notch, at $s = \pm\mathrm{i}\omega_{z}$
> - **$Q_{n}$** — the quality factor of the poles of the notch: the higher it is, the narrower the notch

**What to look for.** The two zeros lie exactly on the imaginary axis and the poles lie just to their left. At the zero $|G|$ is zero. One decade away it is almost one. A zero on the axis removes the frequency and leaves the rest.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 9 · The output as a product

Task 6 found the output of a random drive by a convolution. Lecture 8 says that a convolution in time is a product in frequency. This is the use of $G$.

**Goal.** Compute the same output with no integration and no convolution: multiply the transform of the drive by $G$.

**Needs.** `oscillator` from task 7 and `g_kick` from task 4.

**Steps.**

1. Add zeros to `X_NOISE` until it has `N_PAD` = 8 times as many samples. Take its Fourier transform with `np.fft.rfft(X_NOISE, n=N_PAD)`. The frequencies are `np.fft.rfftfreq(N_PAD, d=1/FS)`. The transform is the subject of `H03`. Here it is a tool.
2. Evaluate $G(\mathrm{i}\omega)$ at those frequencies with `signal.freqresp`, with $\omega = 2\pi f$.
3. Multiply the two. Go back to time with `np.fft.irfft(..., n=N_PAD)` and keep the first samples as long as `T_NOISE`. Call the result `y_product`.
4. The exact answer for this drive is `signal.lsim(oscillator(Q), X_NOISE, T_NOISE)`. Print the largest difference of `y_product` from it, divided by its rms. Print the same number for the convolution `np.convolve(g_kick, X_NOISE)` times $1/f_{s}$.
5. Repeat step 3 with `N_PAD` equal to the number of samples. Print the same difference.
6. Plot the exact output and `y_product` for the first 4 s.

> **The symbols**
>
> - **$X(\omega)$** — the Fourier transform of the drive. The output is $Y(\omega) = G(\mathrm{i}\omega)\,X(\omega)$
> - **`N_PAD`** — the length of the padded drive, in samples

**What to look for.** The product agrees with the exact output. The three routes are integration, convolution and product. Only the last has no loop in time. Without the zeros the product is wrong: the tail of the ringing runs past the end of the record and comes back at the start. We will use this product for filters and for the suspension.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Part D · Stability

The output of a system is a product with $G$, so $G$ decides what the system can do. Its poles decide whether the ringing dies. This is where we go on in your own time.

### Task 10 · Left, on and right of the axis

The real part of a pole is a rate. We put the poles on each side of the imaginary axis and watch the kick.

**Goal.** See the impulse response for poles in the left half plane, on the axis and in the right half plane.

**Needs.** `oscillator` from task 7.

**Steps.**

1. Build the oscillator for $Q = 30$, for no damping at all (`np.inf`) and for $Q = -30$. A negative $Q$ is anti-damping: the motion feeds itself.
2. Compute the impulse response of each for $0 \le t \le 60$ s with `signal.impulse` and plot the three in three panels.
3. Write `ringdown_rate(t, g)`. It finds the peaks of $g$ with `signal.find_peaks`, keeps those above $10^{-3}$ of the largest one and fits a straight line to the logarithm of their heights. It returns the slope. Task 15 uses it.
4. For each case print the real part of the first pole, the rate of the peaks and the largest $|g|$ in the first 10 s and in the last 10 s.
5. For $Q = 30$ print $-1/\mathrm{Re}\,p$ next to $\tau_{0}$.

**What to look for.** The real part of a pole is the rate. Negative real part: the ringing decays. Zero: it never dies. Positive: it grows without bound. The fitted rate of the peaks is the real part of the pole. This is the rule of lecture 8: a system is stable when every pole has $\mathrm{Re}\,p < 0$.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Part E · The Bode plot and the resonance

Stability is about time. A detector is also described in frequency. The Bode plot shows $G$ at real frequencies, which is where we measure it.

### Task 11 · The Bode plot

The poles give the time behaviour. For a stable system $G(\mathrm{i}\omega)$ gives the answer to a sinusoid. The Bode plot is its picture.

**Goal.** Draw the Bode plot of the oscillator for several $Q$ and read the resonance from it.

**Needs.** `oscillator` from task 7.

**Steps.**

1. For $Q = 2$, $10$ and $30$ compute the magnitude in decibels and the phase in degrees on 6000 frequencies between 0.01 and 10 Hz with `signal.bode`. Plot magnitude and phase in two panels with a logarithmic frequency axis.
2. For each $Q$ print four numbers. The peak height divided by the response at 0.01 Hz. The width of the peak, measured between the two frequencies where $|G|$ is the peak divided by $\sqrt{2}$. The phase at $f_{0}$. The slope above resonance, in decibels per decade, between 10 and 100 Hz.

> **The symbols**
>
> - **decibel** — $20\log_{10}|G|$: a factor of 10 in amplitude is 20 dB, so the gains of systems in series add
> - **width** — the full width of the peak where $|G|$ has fallen to $1/\sqrt{2}$ of its maximum

**What to look for.** Four things. The peak is $Q$ times $|G|$ at low frequency. The width is about $f_{0}/Q$. We quote this result and do not derive it. The numbers check it: it is good for $Q = 10$ and $30$ and rough for $Q = 2$. The phase is $-90°$ at $f_{0}$ and goes from $0°$ to $-180°$. Above resonance the magnitude falls by $40$ dB per decade.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 12 · Measure it

A real instrument has no formula for $G$. Its transfer function is measured: drive it with a sinusoid and read the answer. We do this on the oscillator and compare with task 11.

**Goal.** Measure the gain and the phase with sinusoidal drives and add the points to the Bode plot.

**Needs.** `oscillator` from task 7 and the curves of task 11 (`f`, `curves`, `Q_BODE`, `bode_colours`).

**Steps.**

1. Write `measure(system, f_drive)`. It drives the system with $\cos\omega t$ from rest with `signal.lsim`, 100 samples per cycle. It waits twelve decay times $\tau_{0}$ and then projects the next twenty cycles on $\cos\omega t$ and $\sin\omega t$. It returns the gain and the phase of the response.
2. Measure the oscillator with the $Q$ of the parameter cell on 31 frequencies from 0.1 to 10 Hz, plus 21 frequencies between 0.9 and 1.1 Hz.
3. Plot the Bode curves of task 11 again and add the measured points on top.
4. Print the largest relative difference between the measured gain and $|G(\mathrm{i}\omega)|$ from `signal.freqresp`. Print the measured gain and phase at $f_{0}$.

> **The symbols**
>
> - **projection** — the two numbers $c = 2\langle y\cos\omega t\rangle$ and $d = 2\langle y\sin\omega t\rangle$, where $\langle\ \rangle$ is the average over the twenty cycles. The gain is $\sqrt{c^{2} + d^{2}}$ and the phase is $\arctan(-d/c)$

**What to look for.** The measured points lie on the curve. You waited for the transient to die, so each point holds only the steady state. Raise `Q` to 300 and the wait grows ten times. A real suspension has $Q$ of $10^{6}$ or more. That is why its transfer function is measured with a swept sine and a lock-in. Nobody waits for each transient.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Part F · Laplace or Fourier

So far the drives had no beginning. Or we waited for the beginning to die. A real drive is switched on at a moment. Then the two transforms of lecture 8 give different answers.

### Task 13 · The step response

The simplest drive with a beginning is a step: off before $t = 0$ and on after.

**Goal.** Compute the step response $y_{\theta}(t)$ and check that its derivative is $g$.

**Needs.** `oscillator` from task 7.

**Steps.**

1. With the oscillator of $Q$ = `Q` compute the step response for $0 \le t \le 60$ s with `signal.step`. Plot it with the line $y = 1$.
2. Print the final value of $y_{\theta}$ and its peak with the time of the peak.
3. Differentiate $y_{\theta}$ with `np.gradient` and print the largest difference from the impulse response $g(t)$ of `signal.impulse`.

> **The symbols**
>
> - **$\theta(t)$** — the unit step: 0 before $t = 0$ and 1 after
> - **$y_{\theta}(t)$** — the step response: the output for the input $\theta(t)$

**What to look for.** The step response goes to $G(0) = 1$ after some ringing. It overshoots first. Its derivative is $g$, as lecture 8 says. A step is easier to make than a kick, so this is a way to measure $g$.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 14 · A cosine switched on

A cosine that starts at $t = 0$ has a beginning. Laplace knows about it and Fourier does not.

**Goal.** Compare the output that Laplace gives with the one that $G(\mathrm{i}\omega)$ gives. Then decide which tool answers which question.

**Needs.** `oscillator` from task 7 and the time grid `t` from task 13.

**Steps.**

1. Define the drive $\cos\omega t$ for $t \ge 0$ with $f = 1.2$ Hz, on the time grid `t`. Compute the output from rest with `signal.lsim`. This is the Laplace answer: the transient and the steady state together.
2. Compute the Fourier answer: the steady state $|G(\mathrm{i}\omega)|\cos(\omega t + \arg G)$, with $G(\mathrm{i}\omega)$ from `signal.freqresp`.
3. Plot the two outputs in one panel and their difference in a second one. Print both values at $t = 0$.
4. Print the largest difference in the first second and in the one second that starts at $3\tau_{0}$. Print their ratio next to $\mathrm{e}^{-3}$. Print the largest difference after $6\tau_{0}$.

> **The symbols**
>
> - **$\arg G$** — the phase of $G(\mathrm{i}\omega)$

**What to look for.** The Laplace output starts at zero because the system starts at rest. The Fourier output is a cosine that has been there for ever, so it is not zero at $t = 0$. The difference is a transient. It rings at $f_{d}$ and decays as $\mathrm{e}^{-t/\tau_{0}}$, so the poles set it. After a few $\tau_{0}$ the two curves agree. Use Laplace for a signal that starts at a known time and for transients. Use Fourier for a signal with no beginning, such as stationary noise.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## Part G · Applications

We now have a transfer function, its poles and zeros, its stability and its measurement. Part G uses them to build four things: a control loop, a low-pass filter, a notch filter and the suspension of a mirror.

### Task 15 · Control: moving the poles

A suspension keeps its high $Q$ where we measure. Its resonance is damped by a loop that moves the poles to the left.

**Goal.** Close a feedback loop on the velocity and see the poles move. Find the sign that makes it unstable.

**Needs.** `oscillator` from task 7 and `ringdown_rate` from task 10.

**Steps.**

1. A controller measures the velocity and pushes the mass back with the force per unit mass $-\beta\,\omega_{0}\,\dot{y}$. Each derivative becomes a factor $s$, so the denominator of $G$ becomes $s^{2} + \omega_{0}(1/Q + \beta)\,s + \omega_{0}^{2}$. Write `closed_loop(beta)` that returns this transfer function, with $Q$ = `Q`.
2. Sweep $\beta$ from $-0.06$ to $2$ in 700 steps. For each, find the two poles with `np.roots`. Draw them all as small dots in the $s$ plane.
3. Mark the poles for $\beta = 0$, $0.1$, $0.4$ and $-0.05$ with crosses.
4. For those four gains plot $g(t)$ for $0 \le t \le 60$ s in four panels.
5. For each print $Q_{\mathrm{cl}}$, the real part of the pole, the time $-1/\mathrm{Re}\,p$ and the rate of the peaks from `ringdown_rate`.

> **The symbols**
>
> - **$\beta$** — the feedback gain, a number without a unit
> - **$Q_{\mathrm{cl}} = 1/(1/Q + \beta)$** — the quality factor with the loop closed

**What to look for.** Feedback with the right sign pushes the poles to the left and the ring-down gets shorter. With the wrong sign it pushes them to the right. Past $\beta = -1/Q$ the poles cross the axis and the system is unstable. This is the idea behind the active damping of a suspension. A cable on the wrong way in a real loop is exactly this.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 16 · A low-pass filter from poles

A filter is a transfer function chosen on purpose. The first choice is poles only.

**Goal.** Build Butterworth low-pass filters and use one to remove noise from a slow signal.

**Steps.**

1. For orders $n = 2$, $4$ and $8$ build the analog Butterworth low-pass with a cutoff of 5 Hz, `F_CUT = 5.0`. `signal.butter(n, WC, btype='low', analog=True)` returns the two polynomials. Make a `TransferFunction` from them.
2. For each order print the number of zeros and of poles. Print the smallest and the largest $|p|/\omega_{c}$ and the largest $\mathrm{Re}\,p/\omega_{c}$. Print $20\log_{10}|G|$ at $\omega_{c}$ and the slope in decibels per decade between $10\omega_{c}$ and $100\omega_{c}$.
3. Draw the poles of the three filters in the $s$ plane. Draw the Bode magnitude of the three filters on one plot.
4. Make 20 s of a 1 Hz sine of amplitude 1 plus Gaussian noise of rms 1, sampled at 1000 Hz. Filter the sum with the order 4 filter, with `signal.lsim`. Plot three seconds of the noisy data, the sine alone and the output.
5. Ignore the first two seconds. Print the rms of the noise before and after the filter. Print $|G|$ at 1 Hz and the delay it causes, $-\arg G/\omega$.

> **The symbols**
>
> - **$n$** — the order of the filter: the number of poles
> - **$\omega_{c} = 2\pi f_{c}$** — the cutoff: the frequency where the gain has fallen by 3 dB

**What to look for.** The poles of a Butterworth filter lie on a circle of radius $\omega_{c}$, all on the left. A filter of order $n$ falls by $20n$ dB per decade, so more poles give a steeper edge. The low-pass keeps the 1 Hz sine with a small delay and cuts the noise rms by a large factor. This delay is the group delay of lecture 8: the price of a causal filter.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 17 · A notch on a line

The second choice is zeros. Detector data carry lines from the mains. A zero on the axis at the line removes it.

**Goal.** Remove a 50 Hz line from data with the notch of task 8 and check that a nearby signal survives.

**Needs.** `notch` from task 8.

**Steps.**

1. Make 20 s of data sampled at 1000 Hz: a 30 Hz sine of amplitude 0.5, a 50 Hz line of amplitude 5 and Gaussian noise of rms 0.5.
2. Filter the data with `notch(F_LINE, Q_NOTCH)` and `signal.lsim`.
3. Write `amplitude_at(x, frequency)`. It ignores the first two seconds, projects $x$ on $\mathrm{e}^{-\mathrm{i}\omega t}$ and returns twice the modulus of the average.
4. Print the amplitude at 50 Hz and at 30 Hz before and after the notch. Compare the change at 30 Hz with $|G|$ at 30 Hz.
5. Plot the Bode magnitude of the notch from 20 to 80 Hz. Plot 0.2 s of the data before and after, on a second panel.

**What to look for.** The notch removes the line almost completely. The 30 Hz tone changes by the gain that $|G|$ predicts, about 0.4%. A frequency that we choose is removed. This is the use of zeros on the axis. The small rest at 50 Hz comes from the straight line that `lsim` draws between the samples.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


### Task 18 · Why the mirrors hang

Task 9 turned the output into a product with $G$. The product of two transfer functions is the transfer function of the two systems in a chain. A suspension is such a chain.

**Goal.** Count the pendulum stages that a mirror needs. Each stage is one transfer function. A chain of stages is their product.

A suspension stage is the oscillator of the first tasks with the damping of a fibre. The loss comes from the material and not from the air, so it is described by a loss angle $\phi$. We quote the ground-to-mass transfer of one stage (Saulson 2017):

$$
T(\omega) = \frac{\omega_{p}^{2}(1 + \mathrm{i}\phi)}
{\omega_{p}^{2}(1 + \mathrm{i}\phi) - \omega^{2}} ,
\qquad |T| \to \left(\frac{f_{p}}{f}\right)^{2}\ \text{for } f \gg f_{p} .
$$

> **The symbols**
>
> - **$T(\omega)$** — the transfer function of one stage, from the motion of its support to the motion of its mass
> - **$\omega_{p} = 2\pi f_{p}$** — the resonance of one pendulum stage
> - **$\phi$** — the loss angle of the fibre: it plays the role of $1/Q$

**Steps.**

1. Write a function `stage(f)` that returns $|T|$ at the frequency $f$ in hertz, with the parameters at the top of the notebook.
2. Print $|T|$ at 10 Hz next to $(f_{p}/f)^{2}$. Print $|T|$ at $f_{p}$ next to $1/\phi$.
3. Take ground motion of $10^{-7}/f^{2}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$, a reasonable figure for a quiet site above 1 Hz. The transfer functions of stages in a chain multiply, so $n$ stages leave the mirror with ground motion times $|T|^{n}$. Plot this for $n = 0$ to $5$ between 1 and 100 Hz on logarithmic axes.
4. A 4 km interferometer that must reach $h = 10^{-23}/\sqrt{\mathrm{Hz}}$ at 10 Hz needs mirror motion below $hL = 4\times10^{-20}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$. Draw that line.
5. For each $n$ print the motion at 10 Hz and how many times it is the requirement.
6. Measure the slope of the four-stage curve between 10 and 30 Hz. Find the frequency above which it is below the requirement.

**What to look for.** One stage at resonance is the same peak as in task 11, with $Q = 1/\phi$. Above resonance each stage divides the motion by $(f/f_{p})^{2}$. The slope of the four-stage curve is the count of pendulums plus the ground: $2 + 2 \times 4$. Open `H07`, find the seismic term of the noise budget and account for its exponent.

In [ ]:
# your code here

> **Solution**
>
> Worked through in the session. Write your own version first — comparing
> the two is where the value is.


## What to hand in

A notebook that runs from top to bottom and contains:

1. **Part A.** The free decay inside its envelope; the ringing frequency next to $f_{d}$; $Q$ from the slope of the energy.
2. **Part B.** The impulse response against the formula; the residual of the superposition; the random drive solved by integration and by convolution.
3. **Part C.** The poles of the oscillator for four $Q$ in the $s$ plane, with their largest difference from the formula. The zeros and the poles of the notch. The output of the random drive as a product, with its difference from the exact output.
4. **Part D.** The three impulse responses. The fitted rate of the peaks beside the real part of the poles.
5. **Part E.** The Bode plot for three $Q$ and the table of peak height, width, phase and slope. The measured points on top and the largest difference between measurement and $G(\mathrm{i}\omega)$.
6. **Part F.** The step response. The Laplace and the Fourier output for the switched-on cosine and their difference. One sentence on when to use each.
7. **Part G.** The poles of the feedback loop and the four ring-downs, one of them unstable. The poles and the Bode plot of the three Butterworth filters. The noisy signal before and after. The line before and after the notch. The isolation plot with the requirement drawn on it and the slope of the four-stage curve. One sentence that accounts for the exponent 10.

> **Where this goes next**
>
> You have a system described by its poles and zeros. You drew it as a Bode plot and measured it with sinusoids. You got its output as a product and turned it into filters and a suspension. All of it rests on linearity.
>
> `H03` builds the Fourier transform on data instead of on a formula. `H04` then removes the linearity: the same pendulum, swung hard, changes its own frequency and answers at frequencies nobody asked about.

## Software

`numpy` throughout and `scipy.signal` for the systems. `solve_ivp` integrates the equation of the oscillator. `TransferFunction` holds $G(s)$ with its poles and zeros. `impulse` and `step` give the responses. `lsim` gives the output for any input. `bode` and `freqresp` give the transfer function on the frequency axis, $G(\mathrm{i}\omega)$. `butter` gives the Butterworth poles. `np.fft` gives the transform used in task 9. The oscillator, the loss angle and the pendulum isolation are treated as in (Saulson 2017). The transforms follow (Oppenheim et al. 1997).

**References**

- Saulson (2017). *Fundamentals of Interferometric Gravitational Wave Detectors*.
- Oppenheim et al. (1997). *Signals and Systems*.